<a href="https://colab.research.google.com/github/YBenjaminPCondori/Machine-Learning-Gen-AI/blob/main/Occupancy-Monitoring-Conv1D-System-with-Gaussian-Data-Compression/Conv1D_Occupancy_Detection_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conv1D-Based Occupancy Detection from Multisensor Time-Series Data

This notebook presents an end-to-end Conv1D pipeline for occupancy detection using multisensor environmental time-series data. It is structured for research, reproducibility, and publication-ready reporting.

### Problem Formulation

This is a **supervised binary classification** task over **multivariate time-series data**. The goal is to predict whether a room is occupied (`motion = 1`) or unoccupied (`motion = 0`) using a sliding window of past environmental sensor readings (temperature, humidity, light, etc.).

The model used is a **1D Convolutional Neural Network (Conv1D)**, which is well-suited to extracting local temporal patterns (short-range dependencies) from sequential data without the computational overhead of recurrent architectures like LSTMs.

### Pipeline Overview

| Stage | Purpose |
|---|---|
| 1. Imports | Load all libraries for data handling, modelling, and evaluation |
| 2. Preprocessing | Clean, impute, encode, and normalise raw sensor data |
| 3. Windowing | Reshape flat time-series into overlapping fixed-length input sequences |
| 4. Dataset Pipeline | Batch and optimise data delivery to the GPU via `tf.data` |
| 5. Class Weights | Compensate for imbalanced occupied vs. unoccupied samples |
| 6. Model | Define the Conv1D architecture with regularisation |
| 7. Training | Fit the model with early stopping to prevent overfitting |
| 8–9. Evaluation | Measure performance with metrics, confusion matrix, and ROC analysis |

---
## 1. Imports and Environment Setup

### Why these libraries?

- **`pandas` / `numpy`** — Data ingestion, manipulation, and numerical operations. Pandas provides the DataFrame abstraction for tabular sensor data; NumPy provides the underlying array operations and is the expected input format for scikit-learn and TensorFlow.

- **`tensorflow`** — The deep learning framework used to build, compile, train, and evaluate the Conv1D model. TensorFlow's `tf.data` API is also used to create an efficient batched input pipeline.

- **`StandardScaler` (sklearn)** — Performs z-score normalisation (zero mean, unit variance) on each feature. This is critical because Conv1D layers and gradient-based optimisers are sensitive to feature scale — without it, features with large magnitudes (e.g., timestamp in Unix seconds) would dominate gradient updates.

- **`class_weight` (sklearn)** — Computes inverse-frequency class weights to address class imbalance. In occupancy datasets, "unoccupied" samples often vastly outnumber "occupied" ones.

- **`sklearn.metrics`** — Provides the full suite of binary classification evaluation metrics: confusion matrix, precision, recall, F1-score, ROC-AUC, and log loss. These go beyond raw accuracy to give a nuanced picture of model performance, especially under class imbalance.

- **`matplotlib` / `seaborn`** — Visualisation libraries for plotting training curves, confusion matrices, and ROC curves.

In [2]:

import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss, roc_curve
)

import matplotlib.pyplot as plt
import seaborn as sns


---
## 2. Dataset Loading and Preprocessing

### 2.1 Loading the Raw Data

The dataset is a merged CSV combining environmental sensor readings (temperature, humidity, light, CO, smoke) with motion detection labels from multiple smart-home devices. Each row represents a single timestamped sensor reading.

### 2.2 Deduplication and Column Cleaning

- **`drop_duplicates()`** — Removes exact duplicate rows that can arise from overlapping sensor logs or re-merged CSV files. Duplicates would artificially inflate the dataset size and bias the model toward repeated patterns.

- **Column name normalisation** (`.str.strip().str.lower()`) — Ensures consistent lowercase column names with no trailing whitespace. This prevents subtle bugs like `'Motion'` vs `'motion'` vs `' motion'` being treated as different columns.

- **Dropping rows where `motion` is NaN** — The target label must exist for every training sample. Rows without a motion reading cannot be used for supervised learning, so they are removed rather than imputed (imputing labels would introduce label noise).

### 2.3 Missing Value Imputation (Mean Imputation)

Missing numeric feature values are filled with the **column mean**. This is a simple imputation strategy that:

- Preserves the overall feature distribution (mean and variance are approximately maintained).
- Assumes data is **Missing At Random (MAR)** — i.e., the missingness does not depend on the unobserved value itself.

**Trade-off:** For time-series data, mean imputation can introduce artificial discontinuities (a sudden jump to the global mean in the middle of a trend). Forward-fill (`ffill`) or linear interpolation would better preserve temporal continuity, but mean imputation is acceptable here as a baseline approach.

### 2.4 Dropping Irrelevant or Leaky Columns

Several columns are removed because they are either **uninformative**, **redundant**, or risk introducing **data leakage**:

| Column | Reason for Dropping |
|---|---|
| `smoke` | Sparse or constant — most indoor environments have zero smoke readings, providing no discriminative signal for occupancy |
| `device` | A categorical device identifier — encodes *which sensor* recorded the reading, not an environmental feature. Including it risks the model learning device-specific biases rather than generalisable occupancy patterns |
| `co` | Carbon monoxide — similar to smoke, typically near-zero in domestic settings and not a reliable occupancy proxy |
| `hub` | Infrastructure metadata (which smart-home hub), not a sensor measurement |
| `home` | Identifies which household — dropping this forces the model to learn occupancy patterns that generalise across homes rather than memorising per-home baselines |

### 2.5 Timestamp Encoding

The timestamp string is converted to a **Unix epoch** (seconds since 1 January 1970) by parsing to `datetime64` and then dividing nanosecond integers by 1e9.

**Why?** Neural networks require numeric inputs. The Unix timestamp gives the model a monotonically increasing feature that captures the **absolute time** and overall trend of the data.

**Limitation:** A raw epoch does not encode **cyclical patterns** (time of day, day of week) that are strongly correlated with occupancy. Sine/cosine transformations of hour-of-day or day-of-week could capture these periodic signals more effectively.

### 2.6 Boolean-to-Integer Conversion (Light)

If the `light` column is stored as a Python `bool` (True/False), it is cast to `int` (1/0). This is necessary because:

- `StandardScaler` and TensorFlow expect numeric types, not booleans.
- The binary encoding (0 = off, 1 = on) is the correct numeric representation for a binary sensor.

### 2.7 Feature Scaling (Z-Score Normalisation)

All feature columns are standardised using `StandardScaler`, which transforms each feature to have **mean = 0** and **standard deviation = 1**:

$$z = \frac{x - \mu}{\sigma}$$

**Why this matters for Conv1D:**

- Convolutional layers apply shared learned filters across the input. If features are on wildly different scales (e.g., temperature in °C vs. timestamp in billions), the filters will be dominated by the high-magnitude feature, and gradient updates will be unstable.
- Standardisation ensures all features contribute roughly equally to the learned representations, leading to faster and more stable convergence during training.

**Note:** The scaler is fit on the *entire dataset* before splitting, which is a minor form of data leakage (test set statistics influence the scaling). In a production pipeline, the scaler should be fit only on the training set and then applied to the test set.

### 2.8 Feature/Target Extraction

The target variable `motion` is cast to integer (0 or 1), and all remaining columns become the feature matrix `X`. Both are converted to `float32` NumPy arrays — this is the standard dtype for TensorFlow operations and reduces memory usage compared to `float64`.

In [3]:

df = pd.read_csv(r"C:/Users/ybenj/Downloads/ALL_CSVS/merged_env_motion.csv")

df.drop_duplicates(inplace=True)
df.columns = df.columns.str.strip().str.lower()
df = df[df['motion'].notna()]

numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

columns_to_drop = ['smoke', 'device']
if 'co' in df.columns:
    columns_to_drop.append('co')
df.drop(columns=columns_to_drop, errors='ignore', inplace=True)

df['timestamp'] = pd.to_datetime(df['timestamp']).astype('int64') / 1e9

if 'light' in df.columns and df['light'].dtype == bool:
    df['light'] = df['light'].astype(int)

df.drop(columns=['hub', 'home'], inplace=True, errors='ignore')

target = 'motion'
df[target] = df[target].astype(int)
features = [col for col in df.columns if col != target]

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

X = df[features].values.astype(np.float32)
y = df[target].values.astype(np.float32)


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/ybenj/Downloads/ALL_CSVS/merged_env_motion.csv'

---
## 3. Sliding Window Construction

### 3.1 Why Sliding Windows?

A Conv1D model expects input of shape `(batch_size, time_steps, num_features)` — a 3D tensor. The raw data is a 2D table (rows × features), so we need to reshape it into **overlapping fixed-length sequences** (windows).

Each window contains `TIME_STEPS = 100` consecutive timesteps of all sensor features. The model reads this window and predicts the occupancy state at the **next** timestep (i.e., `labels[i + window_size]`). This makes the task a **one-step-ahead prediction**: given the last 100 readings, predict whether the room will be occupied at timestep 101.

### 3.2 How the Windowing Works

```
Raw data:   [t0, t1, t2, t3, ..., t99, t100, t101, ...]

Window 1:   [t0  ... t99]   → label = motion at t100
Window 2:   [t1  ... t100]  → label = motion at t101
Window 3:   [t2  ... t101]  → label = motion at t102
...
```

Adjacent windows overlap by 99 out of 100 timesteps — this is standard practice to maximise the number of training samples, but it means consecutive windows share almost all of their data.

### 3.3 Hyperparameter Choices

| Parameter | Value | Rationale |
|---|---|---|
| `TIME_STEPS` | 100 | Defines the model's **temporal receptive field** — how far back the model can look. The ideal value depends on the sampling rate: at 1 sample/second this is ~1.5 minutes; at 1 sample/minute this is ~1.5 hours. |
| `BATCH_SIZE` | 32 | Standard mini-batch size that balances gradient noise (too small → noisy updates, too large → poor generalisation) and GPU memory usage. |

### 3.4 Output Shape

After windowing, `X_windowed` has shape `(N - 100, 100, num_features)` and `y_windowed` has shape `(N - 100,)`, where `N` is the total number of rows in the cleaned dataset.

In [ ]:

def create_windows(data, labels, window_size):
    X_windows, y_labels = [], []
    for i in range(len(data) - window_size):
        X_windows.append(data[i:i+window_size])
        y_labels.append(labels[i+window_size])
    return np.array(X_windows), np.array(y_labels)

TIME_STEPS = 100
BATCH_SIZE = 32

X_windowed, y_windowed = create_windows(X, y, TIME_STEPS)


---
## 4. TensorFlow Dataset Pipeline

### 4.1 `tf.data.Dataset` — Why Not Just NumPy Arrays?

TensorFlow's `tf.data` API provides a high-performance input pipeline that handles batching, shuffling, and prefetching in a way that overlaps data preparation with model computation. This means the GPU is never idle waiting for the next batch — `prefetch(AUTOTUNE)` lets TensorFlow automatically tune how many batches to prepare in advance.

### 4.2 Shuffling

`.shuffle(1000)` randomises the order of samples using a buffer of 1,000 elements. This ensures the model does not learn from the sequential order of batches during training (e.g., all "occupied" samples arriving first).

### 4.3 Train/Test Split

The dataset is split 80/20 by **batch count** using `.take()` and `.skip()`:

- `train_ds = dataset.take(train_size)` — first 80% of batches
- `test_ds = dataset.skip(train_size)` — remaining 20% of batches

### ⚠️ Important Methodological Note — Data Leakage

The shuffle happens **before** the split, which means temporally adjacent windows (which overlap by 99/100 timesteps) can end up in both train and test sets. This creates **severe data leakage** — the test set contains data nearly identical to training samples, which will **inflate all evaluation metrics**.

For time-series data, the correct approach is a **chronological split** with no shuffle crossing the boundary:

```python
# Correct approach (not used here):
split_idx = int(0.8 * len(X_windowed))
X_train, X_test = X_windowed[:split_idx], X_windowed[split_idx:]
y_train, y_test = y_windowed[:split_idx], y_windowed[split_idx:]
```

This is left as-is for this version of the notebook, but should be addressed in any production or publication pipeline.

In [ ]:

dataset = tf.data.Dataset.from_tensor_slices((X_windowed, y_windowed))
dataset = dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

total_batches = len(list(dataset))
train_size = int(0.8 * total_batches)

train_ds = dataset.take(train_size)
test_ds = dataset.skip(train_size)


---
## 5. Class Imbalance Handling

### 5.1 The Problem — Imbalanced Classes

In most occupancy datasets, rooms are **unoccupied far more often than occupied**. If 90% of samples are class 0 (no motion), a model that always predicts "unoccupied" would achieve 90% accuracy while being completely useless. The model has no incentive to learn the minority class.

### 5.2 The Solution — Inverse-Frequency Class Weights

`compute_class_weight('balanced')` calculates weights inversely proportional to class frequency:

$$w_c = \frac{N}{k \cdot n_c}$$

where $N$ is the total number of samples, $k$ is the number of classes (2), and $n_c$ is the number of samples in class $c$.

For example, if 80% of samples are class 0 and 20% are class 1:
- Weight for class 0: ~0.625 (penalised less per sample)
- Weight for class 1: ~2.5 (penalised more per sample)

These weights are passed to `model.fit()` via the `class_weight` argument, which scales the loss contribution of each sample according to its class.

### 5.3 Why Not Oversampling (SMOTE)?

Resampling techniques like SMOTE generate synthetic samples by interpolating between existing minority-class examples. However, for time-series data this would **break temporal ordering** — the synthetic samples would not correspond to any real temporal context. Cost-sensitive learning (class weights) avoids this problem entirely by operating at the loss-function level.

In [ ]:

class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_windowed.astype(int)),
    y=y_windowed.astype(int)
)

class_weights = dict(enumerate(class_weights_array))


---
## 6. Conv1D Model Architecture

### 6.1 Architecture Overview

The model is a **Sequential Conv1D network** consisting of three convolutional blocks followed by a fully-connected classification head. It follows a standard pattern for 1D temporal classification: extract local features with convolutions, progressively downsample with pooling, then classify with dense layers.

```
Input (100 timesteps × N features)
  │
  ├─ Conv1D(64, kernel=3) + BatchNorm + MaxPool(2)   → 50 × 64
  ├─ Conv1D(128, kernel=3) + BatchNorm + MaxPool(2)  → 25 × 128
  ├─ Conv1D(256, kernel=3) + BatchNorm + MaxPool(2)  → 12 × 256
  │
  ├─ Flatten                                          → 3072
  ├─ Dense(128) + Dropout(0.5)
  ├─ Dense(64) + Dropout(0.5)
  └─ Dense(1, sigmoid)                                → P(occupied)
```

### 6.2 Convolutional Blocks — Feature Extraction

Each block consists of three operations:

**Conv1D(filters, kernel_size=3, padding='same', activation='relu')**
- Slides a 1D filter of width 3 across the time axis, computing a dot product at each position. This captures **local temporal patterns** spanning 3 consecutive timesteps (e.g., a sudden rise in temperature, a brief spike in humidity).
- `padding='same'` pads the input so the output length matches the input length (before pooling).
- `ReLU` activation ($f(x) = \max(0, x)$) introduces nonlinearity and induces sparse activations.
- Filter counts **increase** with depth (64 → 128 → 256) so that deeper layers can represent more complex and abstract feature combinations.

**BatchNormalization()**
- Normalises activations within each mini-batch to have zero mean and unit variance. This stabilises and accelerates training by reducing **internal covariate shift** (the phenomenon where the distribution of layer inputs keeps changing as preceding layers update).

**MaxPooling1D(pool_size=2)**
- Halves the temporal dimension by taking the maximum value in each pair of adjacent positions. This serves two purposes:
  1. **Dimensionality reduction** — reduces computation in subsequent layers.
  2. **Translation invariance** — makes the model slightly invariant to small shifts in when a pattern occurs within the window.
- After 3 pooling layers: 100 → 50 → 25 → 12 timesteps remain.

### 6.3 Receptive Field Analysis

The **effective receptive field** is the span of original timesteps that a single neuron in the deepest convolutional layer can "see":

- With kernel size 3 and 3 layers (each with pooling that doubles the effective stride), the theoretical receptive field is approximately **22–26 timesteps** out of 100.
- This means the Conv1D layers alone cannot capture patterns spanning the entire 100-step window — the Flatten + Dense layers compensate by combining information across all remaining positions.
- **Alternative:** Dilated convolutions (e.g., `dilation_rate=2, 4, 8`) could exponentially expand the receptive field without adding parameters.

### 6.4 Classification Head — Dense Layers

**Flatten()**
- Reshapes the 3D output of the last convolutional block `(batch, 12, 256)` into a 1D vector `(batch, 3072)` so it can be fed into dense layers.

**Dense(128) → Dense(64) → Dense(1)**
- Two hidden dense layers progressively reduce dimensionality (3072 → 128 → 64 → 1), learning nonlinear decision boundaries over the extracted features.

**Dropout(0.5)**
- During training, randomly sets 50% of activations to zero at each forward pass. This is a **regularisation technique** that:
  - Prevents co-adaptation of neurons (forces each neuron to be independently useful).
  - Approximates an ensemble of many sub-networks.
  - 0.5 is aggressive — if underfitting is observed, reducing to 0.3 may help.

**Sigmoid output**
- Maps the final logit to $[0, 1]$, interpreted as $P(\text{occupied})$. A threshold of 0.5 is applied at inference to produce the binary prediction.

### 6.5 Compilation — Loss, Optimiser, and Metrics

**Loss: `binary_crossentropy`**
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i)\right]$$

The standard loss function for binary classification. It penalises confident wrong predictions heavily (via the logarithm), encouraging well-calibrated probability outputs.

**Optimiser: `Adam`**
- Adaptive Moment Estimation — maintains per-parameter adaptive learning rates based on first and second moment estimates of the gradients. It typically converges faster than plain SGD and requires less learning rate tuning.

**Metric: `accuracy`**
- Tracked during training for monitoring. However, under class imbalance, accuracy alone can be misleading — the evaluation section uses more robust metrics (F1, ROC-AUC).

In [ ]:

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(TIME_STEPS, len(features))),
    tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(2)

    tf.keras.layers.Conv1D(128, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(2),

    tf.keras.layers.Conv1D(256, 3, activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

---
## 7. Training with Early Stopping

### 7.1 Early Stopping — Preventing Overfitting

Early stopping monitors the **validation loss** (`val_loss`) at the end of each epoch and halts training if it fails to improve for `patience = 3` consecutive epochs.

- **`monitor='val_loss'`** — We track validation loss rather than training loss because we care about generalisation, not memorisation.
- **`patience=3`** — Allows 3 epochs of no improvement before stopping. This gives the model time to escape shallow local minima while still cutting off before serious overfitting occurs.
- **`restore_best_weights=True`** — After stopping, the model's weights are rolled back to the epoch with the lowest validation loss. Without this, the model would retain the weights from the last (worst) epoch.

### 7.2 Training Configuration

| Parameter | Value | Purpose |
|---|---|---|
| `epochs=50` | Upper bound on training iterations. Early stopping will typically halt well before 50. |
| `validation_data=test_ds` | The held-out test set is used for validation monitoring. (Ideally, a separate validation split would be used to avoid peeking at the test set during training.) |
| `class_weight=class_weights` | The balanced class weights from Section 5, applied per-sample during loss computation. |

### 7.3 What Happens During Each Epoch

1. The model iterates through all batches in `train_ds`, computing forward passes, losses (weighted by `class_weight`), and backpropagation updates.
2. After all training batches, the model evaluates on `test_ds` without updating weights (dropout is disabled, batch norm uses running statistics).
3. The early stopping callback checks whether `val_loss` improved. If not, it increments a counter; if the counter reaches 3, training stops.

In [ ]:

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    epochs=50,
    validation_data=test_ds,
    callbacks=[early_stop],
    class_weight=class_weights
)


---
## 8. Evaluation

### 8.1 Test Set Evaluation

`model.evaluate()` runs a forward pass over the entire test set and returns:

- **Test Loss** — The binary cross-entropy loss on unseen data. Lower is better. This measures how well-calibrated the model's probability outputs are.
- **Test Accuracy** — The proportion of correct predictions. Useful as a quick sanity check, but potentially misleading under class imbalance (e.g., 95% accuracy could mean the model ignores the minority class entirely).

These are aggregate metrics — the next section provides a more granular breakdown.

In [ ]:

loss, accuracy = model.evaluate(test_ds)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")


---
## 9. Metrics and Visualisation

### 9.1 Collecting Predictions

We iterate over the test set batch by batch, collecting three arrays:

- **`y_true`** — The actual ground-truth labels (0 or 1).
- **`y_pred_probs`** — The raw sigmoid output probabilities, $P(\text{occupied})$. These are used for ROC-AUC computation and probability-calibration analysis.
- **`y_pred_classes`** — The binarised predictions using a threshold of 0.5: if $P \geq 0.5$, predict occupied (1); otherwise predict unoccupied (0).

**Note on threshold:** The 0.5 threshold is the default but not necessarily optimal. For imbalanced data, tuning the threshold to maximise F1-score on a validation set (e.g., by scanning the precision-recall curve) often yields better real-world performance.

In [ ]:

y_true, y_pred_probs, y_pred_classes = [], [], []


In [ ]:
for x_batch, y_batch in test_ds:
    preds = model.predict(x_batch).flatten()
    y_true.extend(y_batch.numpy())
    y_pred_probs.extend(preds)
    y_pred_classes.extend((preds > 0.5).astype(int))

y_true = np.array(y_true).astype(int)
y_pred_probs = np.array(y_pred_probs)
y_pred_classes = np.array(y_pred_classes)

### 9.2 Confusion Matrix

The confusion matrix is a 2×2 table showing the counts of all four prediction outcomes:

| | Predicted: 0 (Unoccupied) | Predicted: 1 (Occupied) |
|---|---|---|
| **Actual: 0** | True Negatives (TN) | False Positives (FP) — "false alarm" |
| **Actual: 1** | False Negatives (FN) — "missed detection" | True Positives (TP) |

**Why it matters for occupancy detection:**

- **False Negatives (FN)** are particularly costly — if the system fails to detect an occupied room, it might turn off heating/lighting while someone is present.
- **False Positives (FP)** waste energy by keeping systems on in empty rooms, but this is generally less harmful.

This asymmetry means **recall** (TP / (TP + FN)) is arguably more important than precision for this application.

In [ ]:
cm = confusion_matrix(y_true, y_pred_classes)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.show()